# Project: Investigate a Dataset - Medical Appointment No Shows

## Table of Contents
<ul>
<li><a href="#intro">Introduction</a></li>
<li><a href="#wrangling">Data Wrangling</a></li>
<li><a href="#eda">Exploratory Data Analysis</a></li>
<li><a href="#conclusions">Conclusions</a></li>
</ul>

<a id='intro'></a>
## Introduction

<a id='dataset'></a>
### Dataset Description 

We will be analyzing a dataset containing information about medical appointments made between Nov 10, 2015 and June 8, 2016 in the city of Vitória, Brazil, with the aim of investigating factors that may correlate with missed appointments.

The dataset was sourced from [Kaggle: Medical Appointment No Shows](https://www.kaggle.com/datasets/joniarroba/noshowappointments). It contains data on 100K+ medical appointments with information about each appointment and the patient. For this analysis, some columns have been renamed for clarity.

| column name | description | original name |
|---|---|---|
| `PatientId` | unique identifier for each patient |`PatientId`|
| `AppointmentId` | unique identifier for each appointment |`AppointmentID`|
| `Gender` | Male or Female |`Gender`|
| `SchedulingDate` | date the appointment was created - [reference](https://www.kaggle.com/datasets/joniarroba/noshowappointments/discussion/340475) |`ScheduledDay`|
| `AppointmentDate` | date of the appointment - [reference](https://www.kaggle.com/datasets/joniarroba/noshowappointments/discussion/340475) |`AppointmentDay`|
| `Age` | age of the patient |`Age`|
| `Neighborhood` | neighborhood of Vitória, Brazil where the appointment takes place |`Neighbourhood`|
| `FinancialAssistance` | patient enrolled in [Bolsa Família](https://en.wikipedia.org/wiki/Bolsa_Fam%C3%ADlia) financial assistance program |`Scholarship`|
| `Hypertension` | patient diagnosed with hypertension |`Hipertension`|
| `Diabetes` | patient diagnosed with diabetes |`Diabetes`|
| `Alcoholism` | patient diagnosed with alcoholism |`Alcoholism`|
| `NumHandicaps` | number of handicaps - [reference](https://www.kaggle.com/datasets/joniarroba/noshowappointments/discussion/29699#229356) |`Handcap`|
| `SMS_received` | 1 or more text messages were sent to the patient |`SMS_received`|
| `NoShow` | the patient missed the appointment |`No-show`|


### Question(s) for Analysis
- Do any of a patient's demographic or health factors correlate to a higher likelihood of missing an appointment?
- Do any factors related to the appointment itself correlate to a higher likelihood of a missed appointment?

<a id='wrangling'></a>
## Data Wrangling

### Environment Setup

In [183]:
import numpy as np
import pandas as pd
import statsmodels.api as sm;
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

%config InlineBackend.figure_format = 'svg'
%matplotlib inline

### General Properties
- 110527 records found
- no empty fields found

In [184]:
df = pd.read_csv(
    "./Database_No_show_appointments/noshowappointments-kagglev2-may-2016.csv"
)

df = df.convert_dtypes()

print(f"rows: {df.shape[0]}")
print(f"any na?: {df.isna().sum().any()}")
print(f"any null?: {df.isnull().sum().any()}\n")

print(df.info())

rows: 110527
any na?: False
any null?: False

<class 'pandas.DataFrame'>
RangeIndex: 110527 entries, 0 to 110526
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   PatientId       110527 non-null  Float64
 1   AppointmentID   110527 non-null  Int64  
 2   Gender          110527 non-null  string 
 3   ScheduledDay    110527 non-null  string 
 4   AppointmentDay  110527 non-null  string 
 5   Age             110527 non-null  Int64  
 6   Neighbourhood   110527 non-null  string 
 7   Scholarship     110527 non-null  Int64  
 8   Hipertension    110527 non-null  Int64  
 9   Diabetes        110527 non-null  Int64  
 10  Alcoholism      110527 non-null  Int64  
 11  Handcap         110527 non-null  Int64  
 12  SMS_received    110527 non-null  Int64  
 13  No-show         110527 non-null  string 
dtypes: Float64(1), Int64(8), string(5)
memory usage: 12.8 MB
None


### Data Cleaning
#### Column naming
Renaming columns for clarity - [see Dataset Description](#dataset) for more info
- `AppointmentID` --> `AppointmentId` for consistency
- `ScheduledDay` --> `SchedulingDate` to better reflect data type
- `AppointmentDay` --> `AppointmentDate` for clarity and to reflect data type
- `Neighbourhood` --> `Neigborhood` (US English spelling)
- `Scholarship` --> `FinancialAssistance` for clarity
- `Hipertension` --> `Hypertension` (US English spelling)
- `Handcap` --> `NumHandicaps` for clarity
- `No-show` --> `NoShow` for consitency and Python compatibility

In [185]:
df.rename(
    columns={
        "AppointmentID": "AppointmentId",
        "ScheduledDay": "SchedulingDate",
        "AppointmentDay": "AppointmentDate",
        "Neighbourhood": "Neighborhood",
        "Scholarship": "FinancialAssistance",
        "Hipertension": "Hypertension",
        "Handcap": "NumHandicaps",
        "No-show": "NoShow"
    },
    inplace=True,
)

#### PatientId

There are many appointments with recurring `PatientId` values.

Within each duplicate `PatientId`, there are no observed differences across:
- `Gender`
- `Hypertension`
- `Diabetes`
- `Alcoholism`
- `numHandicaps`
- `Neighborhood`
- `FinancialAssistance`

The dataset will be examined in terms of individual appointments and individual patients, therefore rows with duplicate `PatientId`s will remain.

In [186]:
print(f"unique PatientIds: {df['PatientId'].nunique()}")
print(f"duplicated PatientIds: {df['PatientId'].duplicated().sum()}")
print(f"duplicate PatientIds incl other factors: {df.duplicated(subset=[
        "PatientId",
        "Gender",
        "Hypertension",
        "Diabetes",
        "Alcoholism",
        "NumHandicaps",
        "Neighborhood",
        "FinancialAssistance"
        ]).sum()}")

unique PatientIds: 62299
duplicated PatientIds: 48228
duplicate PatientIds incl other factors: 48228


#### AppointmentId
- no duplicate `AppointmentId`s found
- `AppointmentId` will be used as the DataFrame index

In [187]:
print(f"duplicate AppointmentIds: {df['AppointmentId'].duplicated().sum()}\n")
df.set_index("AppointmentId", inplace=True)

duplicate AppointmentIds: 0



#### Gender
- 2 values found in data: `["F", "M"]` 
- `Gender` converted to categorical datatype `["female", "male"]`

In [188]:
print(df["Gender"].unique())

df["Gender"] = df["Gender"].map({"F": "female", "M": "male"})
gender_categories = df["Gender"].unique()

gender_category = pd.api.types.CategoricalDtype(
    ordered=False, categories=gender_categories)



df["Gender"] = df["Gender"].astype(gender_category)
print()
print(df["Gender"].unique())

<StringArray>
['F', 'M']
Length: 2, dtype: string

['female', 'male']
Categories (2, str): ['female', 'male']


#### SchedulingDate and AppointmentDate
- `SchedulingDate` and `AppointmentDate` converted to `datetime` datatype
- timestamps are GMT
- time of day not recorded for `AppointmentDate`
- time of day was recorded for `SchedulingDate`, but this is not helpful information. Will zero out time of day to enable like-for-like comparison to `AppointmentDate`
- dropping rows where `AppointmentDate` is 1 day or more before `SchedulingDate` 

In [189]:
df["SchedulingDate"] = pd.to_datetime(df["SchedulingDate"]).dt.normalize()
df["AppointmentDate"] = pd.to_datetime(df["AppointmentDate"])

before = df.shape[0]
df = df[df["AppointmentDate"].dt.date >= df["SchedulingDate"].dt.date]
after = df.shape[0]

print(f"dropped {before - after} rows with bad timestamps")


dropped 5 rows with bad timestamps


#### Age
- Droped rows with invalid Age values `Age < 0`
- `Age == 0` assumed to be an infant
- Right-sized data type to cover realistic range of ages

In [190]:
print(f'{df["Age"].nunique()} unique ages: {df["Age"].min()} - {df["Age"].max()}')
df = df[df["Age"] >= 0]
print(f'{df["Age"].nunique()} unique ages: {df["Age"].min()} - {df["Age"].max()}')
df["Age"] = df["Age"].astype("Int8")

104 unique ages: -1 - 115
103 unique ages: 0 - 115


#### Neighborhood

- 81 unique neighborhoods
- converted to categorical datatype

In [191]:
print(f"unique neighborhoods: {df['Neighborhood'].nunique()}")
neighborhood_categories = df["Neighborhood"].unique()

neighborhood_category = pd.api.types.CategoricalDtype(
    ordered=False, categories=neighborhood_categories)

df["Neighborhood"] = df["Neighborhood"].astype(neighborhood_category)

unique neighborhoods: 81


##### Encoded Booleans
Converted binary columns (`[0,1]`, `["Yes","No"]`) to boolean
- `FinancialAssistance`,
- `Hypertension`,
- `Diabetes`,
- `Alcoholism`,
- `SMS_received`,
- `NoShow`

In [192]:
binaryColumns = [
    "FinancialAssistance",
    "Hypertension",
    "Diabetes",
    "Alcoholism",
    "SMS_received",
    "NoShow",
]

for colName in binaryColumns:
    print(f"{df[colName].value_counts()}\n")

df["FinancialAssistance"] = df["FinancialAssistance"].astype("boolean")
df["Hypertension"] = df["Hypertension"].astype("boolean")
df["Diabetes"] = df["Diabetes"].astype("boolean")
df["Alcoholism"] = df["Alcoholism"].astype("boolean")
df["SMS_received"] = df["SMS_received"].astype("boolean")
df["NoShow"] = df["NoShow"].map({"Yes": True, "No": False}).astype("boolean")

FinancialAssistance
0    99660
1    10861
Name: count, dtype: Int64

Hypertension
0    88720
1    21801
Name: count, dtype: Int64

Diabetes
0    102578
1      7943
Name: count, dtype: Int64

Alcoholism
0    107161
1      3360
Name: count, dtype: Int64

SMS_received
0    75039
1    35482
Name: count, dtype: Int64

NoShow
No     88207
Yes    22314
Name: count, dtype: Int64



#### numHandcap
- Right-sized datatype to fit value range

In [193]:
print(df["NumHandicaps"].value_counts())
df["NumHandicaps"] = df["NumHandicaps"].astype("Int8")

NumHandicaps
0    108282
1      2040
2       183
3        13
4         3
Name: count, dtype: Int64


#### Data Cleanup results

In [194]:
df.info()

<class 'pandas.DataFrame'>
Index: 110521 entries, 5642903 to 5629448
Data columns (total 13 columns):
 #   Column               Non-Null Count   Dtype              
---  ------               --------------   -----              
 0   PatientId            110521 non-null  Float64            
 1   Gender               110521 non-null  category           
 2   SchedulingDate       110521 non-null  datetime64[us, UTC]
 3   AppointmentDate      110521 non-null  datetime64[us, UTC]
 4   Age                  110521 non-null  Int8               
 5   Neighborhood         110521 non-null  category           
 6   FinancialAssistance  110521 non-null  boolean            
 7   Hypertension         110521 non-null  boolean            
 8   Diabetes             110521 non-null  boolean            
 9   Alcoholism           110521 non-null  boolean            
 10  NumHandicaps         110521 non-null  Int8               
 11  SMS_received         110521 non-null  boolean            
 12  NoShow     

In [195]:
df.head()
# df.to_csv("No Show Data - cleaned.csv")

,PatientId,Gender,SchedulingDate,AppointmentDate,Age,Neighborhood,FinancialAssistance,Hypertension,Diabetes,Alcoholism,NumHandicaps,SMS_received,NoShow
AppointmentId,,,,,,,,,,,,,
5642903,29872499824296.0,female,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,62,JARDIM DA PENHA,False,True,False,False,0,False,False
5642503,558997776694438.0,male,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,False,False,False,False,0,False,False
5642549,4262962299951.0,female,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,62,MATA DA PRAIA,False,False,False,False,0,False,False
5642828,867951213174.0,female,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,8,PONTAL DE CAMBURI,False,False,False,False,0,False,False
5642494,8841186448183.0,female,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,False,True,True,False,0,False,False


<a id='eda'></a>
## Exploratory Data Analysis

> **Tip**: Now that you've trimmed and cleaned your data, you're ready to move on to exploration. **Compute statistics** and **create visualizations** with the goal of addressing the research questions that you posed in the Introduction section. You should compute the relevant statistics throughout the analysis when an inference is made about the data. Note that at least two or more kinds of plots should be created as part of the exploration, and you must  compare and show trends in the varied visualizations. Remember to utilize the visualizations that the pandas library already has available.



> **Tip**: Investigate the stated question(s) from multiple angles. It is recommended that you be systematic with your approach. Look at one variable at a time, and then follow it up by looking at relationships between variables. You should explore at least three variables in relation to the primary question. This can be an exploratory relationship between three variables of interest, or looking at how two independent variables relate to a single dependent variable of interest. Lastly, you  should perform both single-variable (1d) and multiple-variable (2d) explorations.


### Do any of a patient's demographic or health factors correlate to a higher likelihood of a missed appointment?

#### Patient demographic and health factors
- `Gender`
- `Age`
- `Neighborhood`
- `FinancialAssistance`
- `Hypertension`
- `Diabetes`
- `Alcoholism`
- `NumHandicaps`

#### Distribution of each factor in the dataset


In [196]:
# create full rank dataframe
genderDummies = pd.get_dummies(df["Gender"], dtype=int)
genderDummies.drop(columns=["male"], inplace=True)

nhoodDummies = pd.get_dummies(df["Neighborhood"], prefix="nhood", dtype=int)
baselineNHood = nhoodDummies.columns[0]
nhoodDummies.drop(columns=[baselineNHood], inplace=True)
print(f"baseline Neighborhood (dropped): {baselineNHood}")

df_fullRank = df.join([nhoodDummies, genderDummies])

df_fullRank["FinancialAssistance"] = df_fullRank["FinancialAssistance"].astype(int)
df_fullRank["Hypertension"] = df_fullRank["Hypertension"].astype(int)
df_fullRank["Diabetes"] = df_fullRank["Diabetes"].astype(int)
df_fullRank["Alcoholism"] = df_fullRank["Alcoholism"].astype(int)
df_fullRank["SMS_received"] = df_fullRank["SMS_received"].astype(int)
df_fullRank["NoShow"] = df_fullRank["NoShow"].astype(int)

df_fullRank.drop(columns=["Neighborhood", "Gender"], inplace=True)
df_fullRank["intercept"] = 1

df_fullRank.head()

baseline Neighborhood (dropped): nhood_JARDIM DA PENHA


,PatientId,SchedulingDate,AppointmentDate,Age,FinancialAssistance,Hypertension,Diabetes,Alcoholism,NumHandicaps,SMS_received,...,nhood_HORTO,nhood_SEGURANÇA DO LAR,nhood_ILHA DO BOI,nhood_FRADINHOS,nhood_NAZARETH,nhood_AEROPORTO,nhood_ILHAS OCEÂNICAS DE TRINDADE,nhood_PARQUE INDUSTRIAL,female,intercept
AppointmentId,,,,,,,,,,,,,,,,,,,,,
5642903,29872499824296.0,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,62,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
5642503,558997776694438.0,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,56,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
5642549,4262962299951.0,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,62,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
5642828,867951213174.0,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,8,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
5642494,8841186448183.0,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,56,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,1,1


In [197]:
# expand date columns
df_fullRank["SchedulingDate_dayName"] = df_fullRank["SchedulingDate"].dt.day_name()
df_fullRank["AppointmentDate_dayName"] = df_fullRank["AppointmentDate"].dt.day_name()
df_fullRank["numDaysCreatedToScheduled"] = (df_fullRank["AppointmentDate"]- df_fullRank["SchedulingDate"]).dt.days

schedulingDayDummies = pd.get_dummies(
    df_fullRank["SchedulingDate_dayName"], prefix="sch", dtype=int
)
baselineSchedulingDay = "sch_Saturday"
schedulingDayDummies.drop(columns=[baselineSchedulingDay], inplace=True)
print(f"baseline SchedulingDate_dayName '{baselineSchedulingDay}' dropped")
print(schedulingDayDummies.columns)

appointDayDummies = pd.get_dummies(
    df_fullRank["AppointmentDate_dayName"], prefix="apt", dtype=int
)
baselineAppointDay = "apt_Saturday"
appointDayDummies.drop(columns=[baselineAppointDay], inplace=True)
print(f"baseline AppointmentDate_dayName '{baselineAppointDay}' dropped")
print(appointDayDummies.columns)

df_fullRank = df_fullRank.join([schedulingDayDummies, appointDayDummies])
df_fullRank.drop(columns=["SchedulingDate_dayName", "AppointmentDate_dayName"], inplace=True)

baseline SchedulingDate_dayName 'sch_Saturday' dropped
Index(['sch_Friday', 'sch_Monday', 'sch_Thursday', 'sch_Tuesday',
       'sch_Wednesday'],
      dtype='str')
baseline AppointmentDate_dayName 'apt_Saturday' dropped
Index(['apt_Friday', 'apt_Monday', 'apt_Thursday', 'apt_Tuesday',
       'apt_Wednesday'],
      dtype='str')


In [198]:
# prune duplicate patientIds
allPatients = df_fullRank.drop_duplicates(subset=["PatientId"])

In [199]:
# logistic regression for Patient demographic factors
demographic_model = sm.Logit(allPatients["NoShow"], allPatients[[
    "intercept",
    "FinancialAssistance",
    "Hypertension",
    "Diabetes",
    "Alcoholism",
    "female"
    ]])

demographic_results = demographic_model.fit()
demographic_results.summary()

Optimization terminated successfully.
         Current function value: 0.493242
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                 NoShow   No. Observations:                62298
Model:                          Logit   Df Residuals:                    62292
Method:                           MLE   Df Model:                            5
Date:                Tue, 25 Aug 2026   Pseudo R-squ.:                0.002322
Time:                        11:56:42   Log-Likelihood:                -30728.
converged:                       True   LL-Null:                       -30799.
Covariance Type:            nonrobust   LLR p-value:                 4.071e-29
=======================================================================================
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
intercept              -1.3907      0.018    -78.788      0.000      -1.425      -1.356
FinancialAssistance     0.1890      0.034      5.637      0.000       0.123       0.255
Hypertension           -0.2621      0.030     -8.821      0.000      -0.320      -0.204
Diabetes                0.0041      0.046      0.089      0.929      -0.085       0.093
Alcoholism              0.2497      0.063      3.977      0.000       0.127       0.373
female                  0.0007      0.021      0.031      0.975      -0.041       0.043
=======================================================================================
"""

In [ ]:
# NoShow proportion per neighborhood 0 or 100% will be excluded from logistic regressions
nhoods = df["Neighborhood"].unique()
nhood_NoShow = pd.DataFrame({"nhood": nhoods, "pNoShow": None})

# nhood_NoShow.set_index("nhood", inplace=True)

for nhood in nhood_NoShow["nhood"]:
    nhood_appointments = df[df["Neighborhood"] == nhood]
    numAppointments = nhood_appointments.shape[0]
    numNoShows = nhood_appointments["NoShow"].sum()

    nhoodAppointments = nhood_NoShow[nhood_NoShow["nhood"] == nhood]
    nhood_NoShow.loc[nhood, "pNoShow"] = (
        None
        if numAppointments == 0 or numNoShows == 0 or numNoShows == numAppointments
        else numNoShows / numAppointments
    )

nhood_NoShow.dropna(inplace=True)
nhood_NoShow = nhood_NoShow.apply(lambda x: "nhood_"+x.astype(str), axis=0)

nhood_NoShow

,pNoShow
nhood,
JARDIM DA PENHA,nhood_0.16275470724787205
MATA DA PRAIA,nhood_0.17080745341614906
PONTAL DE CAMBURI,nhood_0.17391304347826086
REPÚBLICA,nhood_0.17125748502994012
GOIABEIRAS,nhood_0.1957142857142857
...,...
SEGURANÇA DO LAR,nhood_0.19310344827586207
ILHA DO BOI,nhood_0.08571428571428572
FRADINHOS,nhood_0.18604651162790697


In [238]:
# logistic regression for neighborhood (all Patients)
patient_nhood_model = sm.Logit(allPatients[["NoShow"], nhood_NoShow.index.tolist()])
patient_nhood_model_results = patient_nhood_model.fit()
patient_nhood_model_results.summary()

InvalidIndexError: (['NoShow'], ['JARDIM DA PENHA', 'MATA DA PRAIA', 'PONTAL DE CAMBURI', 'REPÚBLICA', 'GOIABEIRAS', 'ANDORINHAS', 'CONQUISTA', 'NOVA PALESTINA', 'DA PENHA', 'TABUAZEIRO', 'BENTO FERREIRA', 'SÃO PEDRO', 'SANTA MARTHA', 'SÃO CRISTÓVÃO', 'MARUÍPE', 'GRANDE VITÓRIA', 'SÃO BENEDITO', 'ILHA DAS CAIEIRAS', 'SANTO ANDRÉ', 'SOLON BORGES', 'BONFIM', 'JARDIM CAMBURI', 'MARIA ORTIZ', 'JABOUR', 'ANTÔNIO HONÓRIO', 'RESISTÊNCIA', 'ILHA DE SANTA MARIA', 'JUCUTUQUARA', 'MONTE BELO', 'MÁRIO CYPRESTE', 'SANTO ANTÔNIO', 'BELA VISTA', 'PRAIA DO SUÁ', 'SANTA HELENA', 'ITARARÉ', 'INHANGUETÁ', 'UNIVERSITÁRIO', 'SÃO JOSÉ', 'REDENÇÃO', 'SANTA CLARA', 'CENTRO', 'PARQUE MOSCOSO', 'DO MOSCOSO', 'SANTOS DUMONT', 'CARATOÍRA', 'ARIOVALDO FAVALESSA', 'ILHA DO FRADE', 'GURIGICA', 'JOANA D´ARC', 'CONSOLAÇÃO', 'PRAIA DO CANTO', 'BOA VISTA', 'MORADA DE CAMBURI', 'SANTA LUÍZA', 'SANTA LÚCIA', 'BARRO VERMELHO', 'ESTRELINHA', 'FORTE SÃO JOÃO', 'FONTE GRANDE', 'ENSEADA DO SUÁ', 'SANTOS REIS', 'PIEDADE', 'JESU

In [ ]:
# logistic regression for neighborhood (all Patients)
X = allPatients[["intercept"] + nhoodDummies.columns.tolist()].astype(float)
y = allPatients["NoShow"].astype(float)

patient_neighborhood_model = sm.Logit(y, X)
patient_neighborhood_model_results = patient_neighborhood_model.fit_regularized(
    method="l1",
    alpha=1.0,
    maxiter=1000
)

patient_neighborhood_model_results.summary()

Optimization terminated successfully    (Exit mode 0)
            Current function value: 0.4917322457987625
            Iterations: 677
            Function evaluations: 677
            Gradient evaluations: 677


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                 NoShow   No. Observations:                62298
Model:                          Logit   Df Residuals:                    62232
Method:                           MLE   Df Model:                           65
Date:                Tue, 25 Aug 2026   Pseudo R-squ.:                0.005740
Time:                        12:14:08   Log-Likelihood:                -30623.
converged:                       True   LL-Null:                       -30799.
Covariance Type:            nonrobust   LLR p-value:                 2.748e-41
=====================================================================================================
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
intercept                            -1.5466      0.030    -51.688      0.000      -1.605      -1.488
nhood_MATA DA PRAIA                  -0.0722      0.140     -0.516      0.606      -0.347       0.202
nhood_PONTAL DE CAMBURI                    0        nan        nan        nan         nan         nan
nhood_REPÚBLICA                      -0.2302      0.135     -1.699      0.089      -0.496       0.035
nhood_GOIABEIRAS                     -0.0839      0.138     -0.609      0.543      -0.354       0.186
nhood_ANDORINHAS                      0.1899      0.079      2.389      0.017       0.034       0.346
nhood_CONQUISTA                      -0.0036      0.122     -0.029      0.977      -0.242       0.235
nhood_NOVA PALESTINA                 -0.0172      0.077     -0.222      0.824      -0.169       0.134
nhood_DA PENHA                             0        nan        nan        nan         nan         nan
nhood_TABUAZEIRO                     -0.0758      0.069     -1.093      0.274      -0.212       0.060
nhood_BENTO FERREIRA                  0.1780      0.116      1.536      0.125      -0.049       0.405
nhood_SÃO PEDRO                       0.2296      0.073      3.136      0.002       0.086       0.373
nhood_SANTA MARTHA                   -0.2385      0.074     -3.201      0.001      -0.384      -0.092
nhood_SÃO CRISTÓVÃO                   0.1195      0.082      1.449      0.147      -0.042       0.281
nhood_MARUÍPE                         0.1948      0.082      2.366      0.018       0.033       0.356
nhood_GRANDE VITÓRIA                  0.0939      0.106      0.888      0.375      -0.113       0.301
nhood_SÃO BENEDITO                   -0.0314      0.095     -0.331      0.741      -0.217       0.154
nhood_ILHA DAS CAIEIRAS               0.2859      0.106      2.706      0.007       0.079       0.493
nhood_SANTO ANDRÉ                    -0.1205      0.080     -1.515      0.130      -0.276       0.035
nhood_SOLON BORGES                    0.0048      0.160      0.030      0.976      -0.309       0.319
nhood_BONFIM                          0.1986      0.068      2.903      0.004       0.065       0.333
nhood_JARDIM CAMBURI                  0.0550      0.050      1.104      0.270      -0.043       0.153
nhood_MARIA ORTIZ                     0.1989      0.052      3.809      0.000       0.097       0.301
nhood_JABOUR                               0        nan        nan        nan         nan         nan
nhood_ANTÔNIO HONÓRIO                -0.1991      0.215     -0.928      0.354      -0.620       0.222
nhood_RESISTÊNCIA                     0.1429      0.060      2.396      0.017       0.026       0.260
nhood_ILHA DE SANTA MARIA             0.3145      0.082      3.839      0.000       0.154       0.475
nhood_JUCUTUQUARA                     0.2706      0.131      2.070      0.038       0.014       0.527
nhood_MONTE BELO                      0.2190      0.116      1.890      0.059  

In [ ]:
# logistic regression for neighborhood (all appointments)
X = df_fullRank[["intercept"] + nhoodDummies.columns.tolist()].astype(float)
y = df_fullRank["NoShow"].astype(float)

neighborhood_model = sm.Logit(y, X)
neighborhood_model_results = neighborhood_model.fit_regularized(
    method="l1",
    alpha=1.0,
    maxiter=1000
)

neighborhood_model_results.summary()

Optimization terminated successfully    (Exit mode 0)
            Current function value: 0.5010186718030561
            Iterations: 728
            Function evaluations: 728
            Gradient evaluations: 728


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                 NoShow   No. Observations:               110521
Model:                          Logit   Df Residuals:                   110454
Method:                           MLE   Df Model:                           66
Date:                Tue, 25 Aug 2026   Pseudo R-squ.:                0.004018
Time:                        12:14:30   Log-Likelihood:                -55371.
converged:                       True   LL-Null:                       -55595.
Covariance Type:            nonrobust   LLR p-value:                 6.371e-58
=====================================================================================================
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
intercept                            -1.5444      0.020    -76.297      0.000      -1.584      -1.505
nhood_MATA DA PRAIA                        0        nan        nan        nan         nan         nan
nhood_PONTAL DE CAMBURI                    0        nan        nan        nan         nan         nan
nhood_REPÚBLICA                      -0.0239      0.094     -0.255      0.799      -0.208       0.160
nhood_GOIABEIRAS                           0        nan        nan        nan         nan         nan
nhood_ANDORINHAS                      0.3354      0.054      6.220      0.000       0.230       0.441
nhood_CONQUISTA                       0.0765      0.090      0.848      0.397      -0.100       0.253
nhood_NOVA PALESTINA                  0.0084      0.059      0.143      0.887      -0.107       0.123
nhood_DA PENHA                        0.1141      0.057      1.984      0.047       0.001       0.227
nhood_TABUAZEIRO                      0.0440      0.051      0.871      0.384      -0.055       0.143
nhood_BENTO FERREIRA                  0.3006      0.084      3.563      0.000       0.135       0.466
nhood_SÃO PEDRO                       0.2192      0.054      4.090      0.000       0.114       0.324
nhood_SANTA MARTHA                   -0.1233      0.053     -2.330      0.020      -0.227      -0.020
nhood_SÃO CRISTÓVÃO                        0        nan        nan        nan         nan         nan
nhood_MARUÍPE                         0.2926      0.059      4.981      0.000       0.177       0.408
nhood_GRANDE VITÓRIA                  0.1685      0.079      2.138      0.033       0.014       0.323
nhood_SÃO BENEDITO                         0        nan        nan        nan         nan         nan
nhood_ILHA DAS CAIEIRAS               0.2698      0.077      3.520      0.000       0.120       0.420
nhood_SANTO ANDRÉ                          0        nan        nan        nan         nan         nan
nhood_SOLON BORGES                   -0.1961      0.131     -1.495      0.135      -0.453       0.061
nhood_BONFIM                          0.1453      0.052      2.807      0.005       0.044       0.247
nhood_JARDIM CAMBURI                  0.0925      0.035      2.612      0.009       0.023       0.162
nhood_MARIA ORTIZ                     0.2183      0.038      5.736      0.000       0.144       0.293
nhood_JABOUR                          0.0236      0.056      0.423      0.672      -0.086       0.133
nhood_ANTÔNIO HONÓRIO                 0.0331      0.159      0.208      0.835      -0.279       0.345
nhood_RESISTÊNCIA                     0.1833      0.042      4.320      0.000       0.100       0.266
nhood_ILHA DE SANTA MARIA             0.1007      0.062      1.624      0.104      -0.021       0.222
nhood_JUCUTUQUARA                     0.0957      0.099      0.968      0.333      -0.098       0.289
nhood_MONTE BELO                      0.1596      0.089      1.785      0.074  

In [ ]:
# graphing helper functions
# def patientDistribution(patients: pd.DataFrame) -> dict[str, list[float]]:
#     numUniquePatients = patients.shape[0]
#     numFemale = patients.query("Gender == 'female'").shape[0]
#     numMale = patients.query("Gender == 'male'").shape[0]
#     numFinAssist = patients.query("FinancialAssistance == True").shape[0]
#     numHypertension = patients.query("Hypertension == True").shape[0]
#     numDiabetes = patients.query("Diabetes == True").shape[0]
#     numAlcoholism = patients.query("Alcoholism == True").shape[0]

#     return {
#         "All Patients": [numUniquePatients/allPatients.shape[0]],
#         "Female": [numFemale/numUniquePatients],
#         "Male": [numMale/numUniquePatients],
#         "FinancialAssistance": [numFinAssist/numUniquePatients],
#         "Hypertension": [numHypertension/numUniquePatients],
#         "Diabetes": [numDiabetes/numUniquePatients],
#         "Alcoholism": [numAlcoholism/numUniquePatients]
#     }

# def graphDistributions(data: dict[str, list[float]], title: str) -> tuple[plt.Figure, plt.Axes]:
#     fig, ax = plt.subplots()
#     ax.set_title(title)
#     ax.set_xlim(left=0, right=1)
#     ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))

#     for label, numPatients in data.items():
#         p = ax.barh(label, numPatients, 0.8)
#         ax.bar_label(p, fmt=lambda x: f"{x * 100:.1f}%", label_type="edge", padding=3)

#     return [fig, ax]

In [ ]:
# allPatientDemo = patientDistribution(allPatients)
# showPatientDemo = patientDistribution(showPatients)
# noShowPatientDemo = patientDistribution(noShowPatients)

In [ ]:
# groupNames = (
#     "All Patients",
#     "Female",
#     "Male",
#     "FinancialAssistance",
#     "Hypertension",
#     "Diabetes",
#     "Alcoholism",
# )

# sources = {"All": allPatientDemo, "No Show": noShowPatientDemo, "Show": showPatientDemo}

# demographicDistributions = {
#     label: tuple(demo[k][0] for k in groupNames) for label, demo in sources.items()
# }

# fig, ax = plt.subplots(figsize=(7, 10))
# barContainers = ax.grouped_bar(
#     demographicDistributions,
#     tick_labels=groupNames,
#     group_spacing=1,
#     orientation="horizontal"
# )

# for container in barContainers.bar_containers:
#     ax.bar_label(
#         container, fmt=lambda x: f"{x * 100:.1f}%", label_type="edge", padding=3
#     )

# ax.set_xlabel("% of all Patients")
# ax.legend(loc="upper right", ncols=1)
# ax.set_xlim(0, 1)
# ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))
# ax.tick_params(labelsize="small")
# ax.tick_params("y", labelrotation=90, labelrotation_mode="ytick")

In [ ]:
# Age, NumHandicaps and Neighborhood distributions

# Age histogram - try violinplot?
# dfAgeAll = allPatients["Age"]
# dfAgeShow = showPatients["Age"]
# dfAgeNoShow = noShowPatients["Age"]

# print(allPatients["Age"].max())

# maxAge = allPatients["Age"].max()
# bins = maxAge + 1
# range = (0, maxAge + 1)

# figAge, ((axAgeAll, axAgeShow, axAgeNoShow)) = plt.subplots(
#     nrows=3, ncols=1, figsize=(7, 9), sharey=True
# )

# axAgeAll.hist(x=dfAgeAll, histtype="bar", bins=bins, range=range, density=True)
# axAgeAll.set_title("Age Distribution: All Patients")
# axAgeAll.tick_params(labelsize="small")
# axAgeAll.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))

# axAgeShow.hist(x=dfAgeShow, histtype="bar", bins=bins, range=range, density=True)
# axAgeShow.set_title("Age Distribution: Show Patients")
# axAgeShow.tick_params(labelsize="small")

# axAgeNoShow.hist(
#     x=dfAgeNoShow, histtype="bar", bins=bins, range=range, density=True
# )
# axAgeNoShow.set_title("Age Distribution: No Show Patients")
# axAgeNoShow.tick_params(labelsize="small")

In [ ]:
# NumHandicaps histogram
# dfHandcapAll = allPatients["NumHandicaps"]
# dfHandcapShow = showPatients["NumHandicaps"]
# dfHandcapNoShow = noShowPatients["NumHandicaps"]

# print(dfHandcapAll.value_counts())

# maxNumHandcaps = dfHandcapAll.max()
# bins = np.arange(0.5, maxNumHandcaps + 1.5, 1)
# range = (0, maxNumHandcaps + 1.5)

# figNumHandicaps, ((axHandcapAll, axHandcapShow, axHandcapNoShow)) = plt.subplots(
#     nrows=3, ncols=1, figsize=(7, 9), sharey=True, sharex=True
# )

# countsAll, binsAll, patchesAll = axHandcapAll.hist(
#     x=dfHandcapAll, histtype="bar", bins=bins, range=range, density=True, rwidth=0.8
# )
# axHandcapAll.set_title("NumHandicaps Distribution: All Patients")
# axHandcapAll.tick_params(labelsize="small")
# axHandcapAll.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))
# axHandcapAll.bar_label(patchesAll, fmt=lambda x: f"{x * 100:.1f}%", label_type="edge")
# axHandcapAll.margins(y=0.1)

# countsShow, binsShow, patchesShow = axHandcapShow.hist(
#     x=dfHandcapShow, histtype="bar", bins=bins, range=range, density=True, rwidth=0.8
# )
# axHandcapShow.set_title("NumHandicaps Distribution: Show Patients")
# axHandcapShow.tick_params(labelsize="small")
# axHandcapShow.bar_label(patchesShow, fmt=lambda x: f"{x * 100:.1f}%", label_type="edge")
# axHandcapShow.margins(y=0.1)

# countsNoShow, binsNoShow, patchesNoShow = axHandcapNoShow.hist(
#     x=dfHandcapNoShow, histtype="bar", bins=bins, range=range, density=True, rwidth=0.8
# )
# axHandcapNoShow.set_title("NumHandicaps Distribution: No Show Patients")
# axHandcapNoShow.tick_params(labelsize="small")
# axHandcapNoShow.bar_label(
#     patchesNoShow, fmt=lambda x: f"{x * 100:.1f}%", label_type="edge"
# )
# axHandcapNoShow.margins(0.1)

### Do any factors related to the appointment correlate with a higher likelihood of a missed appointment?

#### Appointment factors
- `SchedulingDate` - day of week
- `AppointmentDate` - day of week
- number of days between `SchedulingDate` and `AppointmentDate`
- `SMS_received`
- number of appointments per `PatientId`
- has patient previously missed an appointment?

In [207]:
# logistic regression: schedule days
schedule_day_model = sm.Logit(
    df_fullRank["NoShow"], df_fullRank[["intercept"] + schedulingDayDummies.columns.tolist()]
)
schedule_day_results = schedule_day_model.fit()
schedule_day_results.summary()

Optimization terminated successfully.
         Current function value: 0.502975
         Iterations 7


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                 NoShow   No. Observations:               110521
Model:                          Logit   Df Residuals:                   110515
Method:                           MLE   Df Model:                            5
Date:                Tue, 25 Aug 2026   Pseudo R-squ.:               9.404e-05
Time:                        12:15:06   Log-Likelihood:                -55589.
converged:                       True   LL-Null:                       -55595.
Covariance Type:            nonrobust   LLR p-value:                   0.06329
=================================================================================
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
intercept        -3.1355      1.022     -3.069      0.002      -5.138      -1.133
sch_Friday        1.7832      1.022      1.745      0.081      -0.219       3.786
sch_Monday        1.7340      1.022      1.697      0.090      -0.268       3.736
sch_Thursday      1.7782      1.022      1.740      0.082      -0.224       3.781
sch_Tuesday       1.7627      1.022      1.725      0.084      -0.240       3.765
sch_Wednesday     1.7554      1.022      1.718      0.086      -0.247       3.758
=================================================================================
"""

In [208]:
# logistic regression: appointment days
appointment_day_model = sm.Logit(df_fullRank["NoShow"], df_fullRank[[
    "intercept",
    "apt_Monday",
    "apt_Tuesday",
    "apt_Wednesday",
    "apt_Thursday",
    "apt_Friday"
]])
appointment_day_results = appointment_day_model.fit()
appointment_day_results.summary()

Optimization terminated successfully.
         Current function value: 0.502898
         Iterations 5


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                 NoShow   No. Observations:               110521
Model:                          Logit   Df Residuals:                   110515
Method:                           MLE   Df Model:                            5
Date:                Tue, 25 Aug 2026   Pseudo R-squ.:               0.0002475
Time:                        12:15:10   Log-Likelihood:                -55581.
converged:                       True   LL-Null:                       -55595.
Covariance Type:            nonrobust   LLR p-value:                 4.521e-05
=================================================================================
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
intercept        -1.2040      0.380     -3.168      0.002      -1.949      -0.459
apt_Monday       -0.1425      0.380     -0.375      0.708      -0.888       0.603
apt_Tuesday      -0.1769      0.380     -0.465      0.642      -0.922       0.569
apt_Wednesday    -0.2021      0.380     -0.531      0.595      -0.948       0.543
apt_Thursday     -0.2235      0.381     -0.587      0.557      -0.969       0.522
apt_Friday       -0.1074      0.380     -0.282      0.778      -0.853       0.638
=================================================================================
"""

In [209]:
# logistic regression for numDaysCreatedToScheduled
numDays_model = sm.Logit(df_fullRank["NoShow"], df_fullRank[["intercept", "numDaysCreatedToScheduled"]])
numDays_results = numDays_model.fit()
numDays_results.summary()

Optimization terminated successfully.
         Current function value: 0.487885
         Iterations 5


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                 NoShow   No. Observations:               110521
Model:                          Logit   Df Residuals:                   110519
Method:                           MLE   Df Model:                            1
Date:                Tue, 25 Aug 2026   Pseudo R-squ.:                 0.03009
Time:                        12:15:27   Log-Likelihood:                -53922.
converged:                       True   LL-Null:                       -55595.
Covariance Type:            nonrobust   LLR p-value:                     0.000
=============================================================================================
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
intercept                    -1.6779      0.010   -175.079      0.000      -1.697      -1.659
numDaysCreatedToScheduled     0.0259      0.000     57.537      0.000       0.025       0.027
=============================================================================================
"""

In [ ]:
# SMS_Recieved
sms_model = sm.Logit(allPatients["NoShow"], allPatients[["intercept", "SMS_recievied"]])
sms_results = sms_model.fit()
sms_results.summary()

In [ ]:
# number of appointments per patient

In [ ]:
# has patient previously missed an appointment

In [ ]:
# Continue to explore the data to address your additional research
#   questions. Add more headers as needed if you have more questions to
#   investigate.

<a id='conclusions'></a>
## Conclusions

> **Tip**: Finally, summarize your findings and the results that have been performed in relation to the question(s) provided at the beginning of the analysis. Summarize the results accurately, and point out where additional research can be done or where additional information could be useful.

> **Tip**: Make sure that you are clear with regards to the limitations of your exploration. You should have at least 1 limitation explained clearly. 

> **Tip**: If you haven't done any statistical tests, do not imply any statistical conclusions. And make sure you avoid implying causation from correlation!

> **Tip**: Once you are satisfied with your work here, check over your report to make sure that it is satisfies all the areas of the rubric (found on the project submission page at the end of the lesson). You should also probably remove all of the "Tips" like this one so that the presentation is as polished as possible.

## Submitting your Project 

> **Tip**: Before you submit your project, you need to create a .html or .pdf version of this notebook in the workspace here. To do that, run the code cell below. If it worked correctly, you should see output that starts with `NbConvertApp] Converting notebook`, and you should see the generated .html file in the workspace directory (click on the orange Jupyter icon in the upper left).

> **Tip**: Alternatively, you can download this report as .html via the **File** > **Download as** submenu, and then manually upload it into the workspace directory by clicking on the orange Jupyter icon in the upper left, then using the Upload button.

> **Tip**: Once you've done this, you can submit your project by clicking on the "Submit Project" button in the lower right here. This will create and submit a zip file with this .ipynb doc and the .html or .pdf version you created. Congratulations!

In [ ]:
# Running this cell will execute a bash command to convert this notebook to an .html file
!python -m nbconvert --to html Investigate_a_Dataset.ipynb